# Part 1 — Python: text, NLP, and a picture-description dataset

### How this notebook works

Minimal Python typing is required. For each task:

1. **A goal** — the target output
2. **A prompt** — copy it into your AI assistant (Claude, ChatGPT, or Colab's built-in Gemini)
3. **An empty cell** — paste the code it returns, run it, check the result
4. **A checkpoint** — the expected output, for verification

On error: **copy the full error message back to the AI** and say
*"this is the error I got, fix it."* That loop accounts for most of the workflow.

The data is **synthetic**, generated by a script, not drawn from real patients.
Nothing entered into a chatbot today is protected health information.

## Setup

Run this cell first.

In [ ]:
# =====================================================================
#  RUN FIRST. Takes ~30 seconds. Wait for the "Ready." line below.
# =====================================================================
# Understanding this cell is not required. Click the play button on the
# left (or press Shift+Enter) and wait. "Ready." confirms setup is complete.
import os, sys, subprocess

REPO = "mjmarte/ai-coding-workshop"   # facilitator fills this in before the workshop

# 1. Download the workshop files, or update a copy from an earlier session.
if os.path.isdir(".git"):
    r = subprocess.run(["git", "pull", "--ff-only", "-q"])
elif os.path.isdir("ai-coding-workshop"):
    os.chdir("ai-coding-workshop")
    r = subprocess.run(["git", "pull", "--ff-only", "-q"])
elif not os.path.exists("data"):
    r = subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git"])
    if r.returncode == 0:
        os.chdir("ai-coding-workshop")
else:
    r = subprocess.run(["git", "status"], capture_output=True)
if r.returncode != 0:
    raise SystemExit(
        "Could not download or update the workshop files. Check the GitHub link "
        "and tell the facilitator.")

# 2. Download the small English model spaCy needs for Task 4
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])
os.makedirs("outputs", exist_ok=True)

# 3. Confirm the data is present before proceeding
missing = [f for f in ("data/transcripts.csv", "data/features.csv", "data/recovery_prediction.csv")
           if not os.path.exists(f)]
if missing:
    raise SystemExit(f"Setup incomplete — missing {missing}. Ask the facilitator.")
print("Ready. Data files:", os.listdir("data"))

---
## Task 1 — Open the data and look at it

**Goal:** load the CSV, check its dimensions, and count participants per group.

Never model data you haven't looked at. This is step one of every analysis.

**Copy this prompt into your AI assistant:**

```
I'm a researcher using Python in a Google Colab notebook. I am a beginner.

I have a CSV file at `data/transcripts.csv` with these columns:
participant_id, group, age, sex, education_years, months_post_onset, wab_aq, transcript

Each row is one person describing a picture out loud. `group` is either "control" or
"aphasia". `wab_aq` is an aphasia severity score from 0-100 (higher = less impaired).
`transcript` is what they said.

Write Python using pandas to:
1. load the file into a DataFrame called `df`
2. print how many rows and columns it has
3. show the first 3 rows
4. count how many people are in each group

Give me just the code, with short comments. Don't explain it afterwards.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> **Checkpoint.** Expected: **60 rows, 8 columns**, 30 participants per group.

---
## Task 2 — Read some actual transcripts

**Goal:** print one control transcript and one transcript from the most impaired participant.

Before building measures of *language*, read the language itself.

**Copy this prompt into your AI assistant:**

```
Using the same `df`, print the transcript of one control participant, and the
transcript of the participant with aphasia who has the LOWEST wab_aq score.
Label each one clearly so I can tell them apart.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> **Checkpoint.** Note the fillers, repeated words, and vague nouns — these are what the next task quantifies.

---
## Task 3 — Your first NLP measures (no fancy library needed)

**Goal:** for every transcript compute: number of words, number of unique words,
type-token ratio (unique / total), and mean word length.

These four measures are arithmetic, not machine learning — much of useful
research NLP is. Reserve heavier tools for when they're actually needed.

**Copy this prompt into your AI assistant:**

```
Add four new columns to `df`, one row per participant, computed from the `transcript` column:

- `n_words`: total number of words
- `n_unique_words`: number of unique words
- `type_token_ratio`: n_unique_words / n_words
- `mean_word_length`: average number of characters per word

Lowercase the text and strip periods before splitting on whitespace. Use a function
plus `.apply()` so it's readable. Then show me participant_id, group, wab_aq, n_words
and type_token_ratio for the first 5 rows.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> **Checkpoint.** Controls should average around 110 words; the aphasia group, around 70.

---
## Task 4 — Part-of-speech tagging with spaCy

**Goal:** classify words by type — content words (nouns, verbs, adjectives,
adverbs) versus fillers ("uh", "um", "well").

This is the first measure requiring a proper NLP library. `content_word_ratio`
is a standard index of speech informativeness.

**Copy this prompt into your AI assistant:**

```
Now use spaCy (the `en_core_web_sm` model, already downloaded) to add two more
columns to `df`:

- `content_word_ratio`: proportion of tokens tagged NOUN, PROPN, VERB, ADJ or ADV,
  excluding anything in this filler list: uh, um, well, hmm, okay, alright, so
- `filler_rate`: proportion of tokens that ARE in that filler list

Ignore punctuation and whitespace tokens when counting. Then show me the mean of both
columns for each group.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> **Checkpoint.** Controls: content-word ratio ≈ 0.41, near-zero fillers. Aphasia: content ≈ 0.39, filler rate substantially higher.

---
## Task 5 — Plot it

**Goal:** three side-by-side boxplots comparing the groups on words produced,
content-word ratio, and filler rate.

A good task for AI assistance — matplotlib syntax is rarely worth memorizing.

**Copy this prompt into your AI assistant:**

```
Make one matplotlib figure with three side-by-side boxplots comparing the "control"
and "aphasia" groups on: n_words, content_word_ratio, and filler_rate.

Requirements:
- one subplot per measure, with a readable title on each
- overlay the individual data points with a little horizontal jitter
- colour the control boxes green and the aphasia boxes pink
- remove the top and right spines
- give the whole figure a title
- save it to outputs/python_figure.png at dpi=150

Use `ax.set_xticks` / `ax.set_xticklabels` for the group labels rather than the
`labels=` argument to boxplot, which is deprecated.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> **Checkpoint.** If the first attempt is ugly, don't fix it by hand — describe the problem to the AI and ask again. That iteration is the skill being practiced.

---
## Task 6 — Which measure actually tracks severity?

**Goal:** correlate every measure with WAB-AQ, **within the aphasia group only**.

Controls sit near ceiling (~99) on the WAB. Pooling groups would manufacture a
correlation out of the group difference rather than a real severity relationship.
An AI assistant will not flag this — the analyst has to.

**Copy this prompt into your AI assistant:**

```
Within the aphasia group only, compute the correlation between wab_aq and each of
these five measures: n_words, type_token_ratio, mean_word_length, content_word_ratio,
filler_rate. Print them sorted from most negative to most positive, rounded to 2 dp.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> **Checkpoint.** `filler_rate` should come out around **-0.83** and `n_words` around **+0.60**. `type_token_ratio` is nearly flat — this returns in the R portion.

---
## Task 7 (advanced) — Narrative-content proxy against a reference description

**Goal:** score lexical overlap between each transcript and a complete description of the
picture. This is a transparent proxy for discourse content, not a validated main-concept
analysis.

TF-IDF plus cosine similarity requires no downloads and illustrates one way to compare
texts as vectors. It is not interchangeable with an embedding model or a clinician-scored
discourse measure.

**Copy this prompt into your AI assistant:**

```
I want a measure of how semantically close each transcript is to a complete
description of the scene.

Here is my reference description:

"a boy is standing on a stool reaching into the cookie jar taking cookies the stool is
tipping over his sister reaches up for a cookie the mother stands at the sink washing
dishes the sink is overflowing and water is running onto the floor she does not notice"

Using scikit-learn's TfidfVectorizer (with English stop words removed) and
cosine_similarity, add a column `semantic_similarity` to `df` giving each transcript's
cosine similarity to that reference.

Then print the group means, and the correlation with wab_aq within the aphasia group.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> **Checkpoint.** Controls ≈ 0.50, aphasia ≈ 0.24, and r ≈ **+0.82** with WAB-AQ. Interpret this as lexical overlap with this reference, not as validated semantic scoring.

---
## Task 8 (advanced) — Development-only classification from language measures

**Goal:** a resampled logistic-regression classifier predicting group from five language
measures. This is a development exercise, not diagnostic-model validation.

An AI assistant will readily produce a model with **no cross-validation** and a
suspiciously high accuracy, without flagging the omission.

**Copy this prompt into your AI assistant:**

```
Fit a logistic regression that predicts `group` (aphasia = 1, control = 0) from these
five features: n_words, type_token_ratio, mean_word_length, content_word_ratio, filler_rate.

Important: I want a resampled estimate, not training accuracy. Use repeated stratified
5-fold cross-validation. Standardise the features inside a Pipeline so no information
leaks across folds. Report balanced accuracy and ROC-AUC, each as mean (SD), and state
the majority-class chance level.

Print the mean CV accuracy, the standard deviation, and the chance level.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> **Checkpoint.** Performance should exceed a 0.50 majority-class baseline. The repeated folds are not an external validation cohort. Ask: *'Which claims would require an independent cohort?'*

---
## Task 9 — Hand off to R

**Goal:** write the feature set to a CSV so the R half of the workshop can model it.

A common research pattern: Python for text processing, R for statistics.

**Copy this prompt into your AI assistant:**

```
Save participant_id plus the five feature columns and semantic_similarity to
`outputs/my_features.csv` (no index column). Print the first few rows to confirm.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> **Checkpoint.** `data/features.csv` already ships with the same measures, so the R half works even if this task wasn't finished. Compare the two files for agreement.

---
## Done with Python

This pipeline covered: lexical measures, POS tagging, a semantic measure, a plot, and a classifier.

**Continue to R** for the statistics — open `r/02_r_starter.R` in Posit Cloud.

Retain the `guardrails/` folder: the sourced rules, the statistics rubric, the data-privacy guide, and a one-page review checklist for vetting any AI analysis before trusting its numbers. Consult it when working with real data.